In [9]:
!pip install rapidfuzz sqlalchemy psycopg2-binary

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for psycopg2: filename=psycopg2-2.9.12-cp312-cp312-macosx_10_13_universal2.whl size=244166 sha256=4826780137f1fbb9e6eca10007dd6cdc776e4412b0b2400ebb82010c141d2b99
  Stored in directory: /Users/tiarasabrina/Library/Caches/pip/wheels/04/74/ff/720d90509a7e28eeecdaf470a74ba12afb31f8857624440482
Successfully built psycopg2

[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import requests
import json
import time
from tqdm import tqdm

In [3]:
gold_company = pd.read_excel("/Users/tiarasabrina/Documents/PROJECT/AI/address-parsing/master-data/master-matched/Master_Matching_20260701_1059 (1).xlsx", engine="openpyxl")
gold_company

,Company ID IDX,Company ID BUMN,Nama Perusahaan IDX,Nama Perusahaan BUMN,Nama Perusahaan AHU,Jaro IDX_BUMN,Jaro IDX_AHU,Jaro BUMN_AHU,Sektor,Sub Sektor,...,Kode Emiten (Ticker),Tanggal IPO (Listed),Papan Pencatatan,Alamat Lengkap,Telepon,Fax,Email,Website,NPWP,Pair Category
0,1.001,NaN,PT Adaro Andalan Indonesia Tbk,NaN,PT Adaro Andalan Indonesia,0.0,1.0,0.0,Energi,"Minyak, Gas & Batu Bara",...,AADI,2024-12-05,Utama,Cyber 2 Tower Lantai 26\nJl. H.R. Rasuna Said ...,(021) 2553 3065 | 02125533000,(021) 2553 3066,corsec@adaroindonesia.com,www.adaroindonesia.com,02.433.115.9-091.000,IDX + AHU
1,1.002,NaN,Adaro Australia Pty Ltd,NaN,NaN,0.0,0.0,0.0,Investasi,Investasi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IDX Only
2,1.003,NaN,Adaro Capital Limited,NaN,NaN,0.0,0.0,0.0,Investasi,Investasi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IDX Only
3,1.004,NaN,Adaro International (Singapore) Pte Ltd,NaN,NaN,0.0,0.0,0.0,Perdagangan batubara,Perdagangan batubara,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IDX Only
4,1.005,NaN,Arindo Holdings (Mauritius) Ltd,NaN,NaN,0.0,0.0,0.0,Investasi,Investasi,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IDX Only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
772560,NaN,NaN,NaN,NaN,PT Fatima Trading Companies,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,"Gedung Fancy Mampang, Jl. Mampang Prpt. Raya N...",0881024990317,NaN,NaN,NaN,NaN,AHU Only
772561,NaN,NaN,NaN,NaN,PT Simpul Eventworks Indonesia,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,Jl. Wolter Monginsidi No.107A,087784451067,NaN,NaN,NaN,NaN,AHU Only
772562,NaN,NaN,NaN,NaN,PT Sinar Fajar Nusantara Kilat,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,Jalan Ancol Barat VIII Nomor 1 Kontener Nomor ...,081387119253,NaN,NaN,NaN,NaN,AHU Only
772563,NaN,NaN,NaN,NaN,PT Stasiun Hangat Sejahtera,0.0,0.0,0.0,NaN,NaN,...,NaN,NaN,NaN,"Infiniti Office, Bellezza BSA, 1st Floor Unit ...",082364055502,NaN,NaN,NaN,NaN,AHU Only


In [4]:
gold_company_address = gold_company[["Alamat Lengkap"]]
gold_company_address

,Alamat Lengkap
0,Cyber 2 Tower Lantai 26\nJl. H.R. Rasuna Said ...
1,NaN
2,NaN
3,NaN
4,NaN
...,...
772560,"Gedung Fancy Mampang, Jl. Mampang Prpt. Raya N..."
772561,Jl. Wolter Monginsidi No.107A
772562,Jalan Ancol Barat VIII Nomor 1 Kontener Nomor ...
772563,"Infiniti Office, Bellezza BSA, 1st Floor Unit ..."


In [11]:
import pandas as pd
from sqlalchemy import create_engine
from rapidfuzz import fuzz, process

# ============================================
# 1. LOAD MASTER DATA WILAYAH DARI POSTGRES
# ============================================
engine = create_engine("postgresql://tiarasabrina@localhost:5432/wilayah_db")

wilayah = pd.read_sql("SELECT kode, nama FROM wilayah.wilayah", engine)
wilayah["nama"] = wilayah["nama"].str.upper().str.strip()

# Pisahin per level berdasarkan panjang kode (sesuaikan lagi setelah lo cek format kode aslinya)
def get_level(kode):
    parts = kode.split(".")
    if len(parts) == 1:
        return "provinsi"
    elif len(parts) == 2:
        return "kabkota"
    elif len(parts) == 3:
        return "kecamatan"
    elif len(parts) == 4:
        return "desa"
    return "unknown"

wilayah["level"] = wilayah["kode"].apply(get_level)

df_prov = wilayah[wilayah["level"] == "provinsi"].reset_index(drop=True)
df_kabkota = wilayah[wilayah["level"] == "kabkota"].reset_index(drop=True)
df_kec = wilayah[wilayah["level"] == "kecamatan"].reset_index(drop=True)
df_desa = wilayah[wilayah["level"] == "desa"].reset_index(drop=True)

print(f"Provinsi: {len(df_prov)}, Kab/Kota: {len(df_kabkota)}, Kecamatan: {len(df_kec)}, Desa: {len(df_desa)}")


# ============================================
# 2. LOAD DATA ALAMAT MENTAH
# ============================================
gold_company = pd.read_excel(
    "/Users/tiarasabrina/Documents/PROJECT/AI/address-parsing/master-data/master-matched/Master_Matching_20260701_1059 (1).xlsx",
    engine="openpyxl"
)
gold_company_address = gold_company[["Alamat Lengkap"]].copy()
gold_company_address["alamat_clean"] = (
    gold_company_address["Alamat Lengkap"]
    .astype(str)
    .str.upper()
    .str.strip()
)


# ============================================
# 3. FUZZY MATCH HIERARKIS (provinsi -> kab/kota -> kecamatan -> desa)
# ============================================
def match_hierarchical(alamat, threshold=75):
    result = {
        "provinsi": None, "score_prov": 0,
        "kabkota": None, "score_kabkota": 0,
        "kecamatan": None, "score_kec": 0,
        "desa": None, "score_desa": 0,
    }

    # skip kalau alamat kosong/NaN
    if not isinstance(alamat, str) or alamat.strip() == "" or alamat.upper() == "NAN":
        return result

    # --- Tahap provinsi ---
    match_prov = process.extractOne(
        alamat, df_prov["nama"], scorer=fuzz.partial_ratio, score_cutoff=threshold
    )
    if not match_prov:
        return result

    prov_nama, score_prov, idx_prov = match_prov
    kode_prov = df_prov.loc[idx_prov, "kode"]
    result["provinsi"] = prov_nama
    result["score_prov"] = score_prov

    # --- Tahap kab/kota ---
    kandidat_kabkota = df_kabkota[df_kabkota["kode"].str.startswith(kode_prov + ".")]
    if kandidat_kabkota.empty:
        return result

    match_kabkota = process.extractOne(
        alamat, kandidat_kabkota["nama"], scorer=fuzz.partial_ratio, score_cutoff=threshold
    )
    if not match_kabkota:
        return result

    kabkota_nama, score_kabkota, idx_kabkota = match_kabkota
    kode_kabkota = kandidat_kabkota.loc[idx_kabkota, "kode"]
    result["kabkota"] = kabkota_nama
    result["score_kabkota"] = score_kabkota

    # --- Tahap kecamatan ---
    kandidat_kec = df_kec[df_kec["kode"].str.startswith(kode_kabkota + ".")]
    if kandidat_kec.empty:
        return result

    match_kec = process.extractOne(
        alamat, kandidat_kec["nama"], scorer=fuzz.partial_ratio, score_cutoff=threshold
    )
    if not match_kec:
        return result

    kec_nama, score_kec, idx_kec = match_kec
    kode_kec = kandidat_kec.loc[idx_kec, "kode"]
    result["kecamatan"] = kec_nama
    result["score_kec"] = score_kec

    # --- Tahap desa ---
    kandidat_desa = df_desa[df_desa["kode"].str.startswith(kode_kec + ".")]
    if kandidat_desa.empty:
        return result

    match_desa = process.extractOne(
        alamat, kandidat_desa["nama"], scorer=fuzz.partial_ratio, score_cutoff=threshold
    )
    if match_desa:
        desa_nama, score_desa, idx_desa = match_desa
        result["desa"] = desa_nama
        result["score_desa"] = score_desa

    return result

# ============================================
# 4. APPLY KE SEMUA ROW
# ============================================
from tqdm import tqdm
tqdm.pandas()

results = gold_company_address["alamat_clean"].progress_apply(match_hierarchical)
result_df = pd.DataFrame(results.tolist())

final = pd.concat([gold_company_address, result_df], axis=1)
final.head(20)

Provinsi: 38, Kab/Kota: 514, Kecamatan: 7285, Desa: 83762


100%|██████████| 772565/772565 [01:59<00:00, 6456.87it/s]


,Alamat Lengkap,alamat_clean,provinsi,score_prov,kabkota,score_kabkota,kecamatan,score_kec,desa,score_desa
0,Cyber 2 Tower Lantai 26\nJl. H.R. Rasuna Said ...,CYBER 2 TOWER LANTAI 26\nJL. H.R. RASUNA SAID ...,PAPUA SELATAN,81.818182,NaN,0.0,NaN,0.0,NaN,0.0
1,NaN,NaN,NaN,0.000000,NaN,0.0,NaN,0.0,NaN,0.0
2,NaN,NaN,NaN,0.000000,NaN,0.0,NaN,0.0,NaN,0.0
3,NaN,NaN,NaN,0.000000,NaN,0.0,NaN,0.0,NaN,0.0
4,NaN,NaN,NaN,0.000000,NaN,0.0,NaN,0.0,NaN,0.0
5,NaN,NaN,NaN,0.000000,NaN,0.0,NaN,0.0,NaN,0.0
6,JL. GURAMI NO. 173,JL. GURAMI NO. 173,NaN,0.000000,NaN,0.0,NaN,0.0,NaN,0.0
7,"MENARA KARYA LANTAI 22, JL. H.R. RASUNA SAID B...","MENARA KARYA LANTAI 22, JL. H.R. RASUNA SAID B...",RIAU,75.000000,NaN,0.0,NaN,0.0,NaN,0.0
8,"GEDUNG CYBER 2 TOWER LT.23, JL. HR RASUNA SAID...","GEDUNG CYBER 2 TOWER LT.23, JL. HR RASUNA SAID...",RIAU,75.000000,NaN,0.0,NaN,0.0,NaN,0.0
9,"GD. MENARA KARYA LT.23, JL.HR. RASUNA SAID BLO...","GD. MENARA KARYA LT.23, JL.HR. RASUNA SAID BLO...",RIAU,75.000000,NaN,0.0,NaN,0.0,NaN,0.0


In [18]:
#final to excel
final.to_excel("/Users/tiarasabrina/Documents/PROJECT/AI/address-parsing/master-data/master-matched/v2_Master_Matching_20260701_1059 (1)_matched.xlsx", index=False)

## TRIAL 2

In [19]:
import re
import pandas as pd
from sqlalchemy import create_engine
from rapidfuzz import fuzz, process
from tqdm import tqdm

tqdm.pandas()

# ============================================
# 1. LOAD MASTER DATA
# ============================================
engine = create_engine("postgresql://tiarasabrina@localhost:5432/wilayah_db")
wilayah = pd.read_sql("SELECT kode, nama FROM wilayah.wilayah", engine)
wilayah["nama"] = wilayah["nama"].str.upper().str.strip()
wilayah = wilayah.dropna(subset=["kode", "nama"])

def get_level(kode):
    parts = kode.split(".")
    return {1: "provinsi", 2: "kabkota", 3: "kecamatan", 4: "desa"}.get(len(parts), "unknown")

wilayah["level"] = wilayah["kode"].apply(get_level)

df_prov    = wilayah[wilayah["level"] == "provinsi"].reset_index(drop=True)
df_kabkota = wilayah[wilayah["level"] == "kabkota"].reset_index(drop=True)
df_kec     = wilayah[wilayah["level"] == "kecamatan"].reset_index(drop=True)
df_desa    = wilayah[wilayah["level"] == "desa"].reset_index(drop=True)


# ============================================
# 2. HELPER: EXACT WORD-BOUNDARY MATCH (bukan asal substring)
# ============================================
def exact_word_match(alamat_upper, candidates_df):
    """
    Cari exact match pakai regex word-boundary, bukan asal 'in' string.
    'RIAU' gak bakal ke-trigger cuma karena ada substring mirip di tengah kata lain.
    Diurutkan dari nama TERPANJANG dulu supaya 'KOTA BANDUNG' kecek duluan
    sebelum 'BANDUNG' doang (menghindari partial match yang kurang spesifik).
    """
    candidates_sorted = candidates_df.assign(
        _len=candidates_df["nama"].str.len()
    ).sort_values("_len", ascending=False)

    for _, row in candidates_sorted.iterrows():
        nama = row["nama"]
        if not nama or len(nama) < 3:  # skip nama terlalu pendek, rawan false positive
            continue
        pattern = r"\b" + re.escape(nama) + r"\b"
        if re.search(pattern, alamat_upper):
            return nama, 100, row["kode"]
    return None, 0, None


# ============================================
# 3. HELPER: FUZZY FALLBACK YANG KETAT
# ============================================
def fuzzy_fallback(alamat_upper, candidates_df, threshold):
    if candidates_df.empty:
        return None, 0, None
    match = process.extractOne(
        alamat_upper, candidates_df["nama"],
        scorer=fuzz.token_set_ratio, score_cutoff=threshold
    )
    if not match:
        return None, 0, None
    nama, score, idx = match
    kode = candidates_df.iloc[idx]["kode"]
    return nama, score, kode


# ============================================
# 4. TOP-DOWN MATCHING
# ============================================
def match_topdown(alamat_upper, threshold_prov=92, threshold_lain=88):
    result = {"provinsi": None, "score_prov": 0, "kode_prov": None,
              "kabkota": None, "score_kabkota": 0, "kode_kabkota": None,
              "kecamatan": None, "score_kec": 0, "kode_kec": None,
              "desa": None, "score_desa": 0, "kode_desa": None}

    # provinsi: exact word-boundary dulu, baru fuzzy ketat
    nama, score, kode = exact_word_match(alamat_upper, df_prov)
    if not nama:
        nama, score, kode = fuzzy_fallback(alamat_upper, df_prov, threshold_prov)
    if not nama:
        return result
    result.update(provinsi=nama, score_prov=score, kode_prov=kode)

    # kabkota: filter di bawah provinsi yang ketemu
    kandidat = df_kabkota[df_kabkota["kode"].str.startswith(kode + ".")]
    nama, score, k = exact_word_match(alamat_upper, kandidat)
    if not nama:
        nama, score, k = fuzzy_fallback(alamat_upper, kandidat, threshold_lain)
    if not nama:
        return result
    result.update(kabkota=nama, score_kabkota=score, kode_kabkota=k)

    # kecamatan
    kandidat = df_kec[df_kec["kode"].str.startswith(k + ".")]
    nama, score, k2 = exact_word_match(alamat_upper, kandidat)
    if not nama:
        nama, score, k2 = fuzzy_fallback(alamat_upper, kandidat, threshold_lain)
    if not nama:
        return result
    result.update(kecamatan=nama, score_kec=score, kode_kec=k2)

    # desa
    kandidat = df_desa[df_desa["kode"].str.startswith(k2 + ".")]
    nama, score, k3 = exact_word_match(alamat_upper, kandidat)
    if not nama:
        nama, score, k3 = fuzzy_fallback(alamat_upper, kandidat, threshold_lain)
    if nama:
        result.update(desa=nama, score_desa=score, kode_desa=k3)

    return result


# ============================================
# 5. BOTTOM-UP FALLBACK (kalau top-down gagal di provinsi)
# ============================================
def match_bottomup(alamat_upper, threshold=90):
    """
    Coba match desa dulu (paling spesifik/unik), baru derive kecamatan/kabkota/
    provinsi dari kode desa yang ketemu. Dipakai HANYA sebagai fallback untuk row
    yang gagal di top-down, karena scan ke semua 75k+ desa lebih berat.
    """
    result = {"provinsi": None, "score_prov": 0, "kode_prov": None,
              "kabkota": None, "score_kabkota": 0, "kode_kabkota": None,
              "kecamatan": None, "score_kec": 0, "kode_kec": None,
              "desa": None, "score_desa": 0, "kode_desa": None}

    nama, score, kode_desa = exact_word_match(alamat_upper, df_desa)
    if not nama:
        nama, score, kode_desa = fuzzy_fallback(alamat_upper, df_desa, threshold)
    if not nama:
        return result  # bener2 gak ketemu apa-apa

    result.update(desa=nama, score_desa=score, kode_desa=kode_desa)

    parts = kode_desa.split(".")
    kode_kec = ".".join(parts[:3])
    kode_kab = ".".join(parts[:2])
    kode_prov = parts[0]

    row_kec = df_kec[df_kec["kode"] == kode_kec]
    row_kab = df_kabkota[df_kabkota["kode"] == kode_kab]
    row_prov = df_prov[df_prov["kode"] == kode_prov]

    if not row_kec.empty:
        result.update(kecamatan=row_kec.iloc[0]["nama"], score_kec=score, kode_kec=kode_kec)
    if not row_kab.empty:
        result.update(kabkota=row_kab.iloc[0]["nama"], score_kabkota=score, kode_kabkota=kode_kab)
    if not row_prov.empty:
        result.update(provinsi=row_prov.iloc[0]["nama"], score_prov=score, kode_prov=kode_prov)

    return result


# ============================================
# 6. MASTER FUNCTION: gabungin top-down + bottom-up fallback
# ============================================
def match_address(alamat):
    if not isinstance(alamat, str) or alamat.strip() == "" or alamat.strip().upper() == "NAN":
        empty = {"provinsi": None, "score_prov": 0, "kode_prov": None,
                  "kabkota": None, "score_kabkota": 0, "kode_kabkota": None,
                  "kecamatan": None, "score_kec": 0, "kode_kec": None,
                  "desa": None, "score_desa": 0, "kode_desa": None,
                  "method": "skipped_empty"}
        return empty

    alamat_upper = alamat.upper().replace("\n", " ")

    result = match_topdown(alamat_upper)
    if result["provinsi"] and result["kabkota"]:
        result["method"] = "topdown"
        return result

    # fallback bottom-up kalau top-down gak lengkap
    result_bu = match_bottomup(alamat_upper)
    if result_bu["desa"] or result_bu["kecamatan"]:
        result_bu["method"] = "bottomup"
        return result_bu

    result["method"] = "no_match"
    return result


# ============================================
# 7. TEST DULU KE SAMPLE KECIL SEBELUM FULL 772K ROWS
# ============================================
sample = gold_company_address["alamat_clean"].dropna().head(50)
sample_results = sample.progress_apply(match_address)
sample_df = pd.concat([sample, pd.DataFrame(sample_results.tolist(), index=sample.index)], axis=1)
sample_df

100%|██████████| 50/50 [02:03<00:00,  2.47s/it]


,alamat_clean,provinsi,score_prov,kode_prov,kabkota,score_kabkota,kode_kabkota,kecamatan,score_kec,kode_kec,desa,score_desa,kode_desa,method
0,CYBER 2 TOWER LANTAI 26\nJL. H.R. RASUNA SAID ...,NaN,0,NaN,NaN,0,NaN,NaN,0,NaN,NaN,0,NaN,no_match
6,JL. GURAMI NO. 173,NaN,0,NaN,NaN,0,NaN,NaN,0,NaN,NaN,0,NaN,no_match
7,"MENARA KARYA LANTAI 22, JL. H.R. RASUNA SAID B...",KALIMANTAN SELATAN,100,63,KABUPATEN BALANGAN,100,63.11,HALONG,100,63.11.02,KARYA,100,63.11.02.2018,bottomup
8,"GEDUNG CYBER 2 TOWER LT.23, JL. HR RASUNA SAID...",LAMPUNG,100,18,KABUPATEN TANGGAMUS,100,18.06,CUKUH BALAK,100,18.06.09,GEDUNG,100,18.06.09.2010,bottomup
9,"GD. MENARA KARYA LT.23, JL.HR. RASUNA SAID BLO...",KALIMANTAN SELATAN,100,63,KABUPATEN BALANGAN,100,63.11,HALONG,100,63.11.02,KARYA,100,63.11.02.2018,bottomup
10,MENARA KARYA LT.23 JL.HR.RASUNA SAID BLOK X-5 ...,KALIMANTAN SELATAN,100,63,KABUPATEN BALANGAN,100,63.11,HALONG,100,63.11.02,KARYA,100,63.11.02.2018,bottomup
11,"GEDUNG MENARA KARYA LT. 18, JL. H.R. RASUNA SA...",LAMPUNG,100,18,KABUPATEN TANGGAMUS,100,18.06,CUKUH BALAK,100,18.06.09,GEDUNG,100,18.06.09.2010,bottomup
12,JL. KOM. LAUT. YOS SUDARSO (DALAM GANG PRIBADI...,ACEH,100,11,KABUPATEN ACEH SELATAN,100,11.01,LABUHANHAJI,100,11.01.04,DALAM,100,11.01.04.2008,bottomup
13,RUKO GREEN GARDEN BLOK A2-10 JL. WAHIDIN SUDIR...,MALUKU UTARA,100,82,KABUPATEN HALMAHERA UTARA,100,82.03,TOBELO UTARA,100,82.03.10,RUKO,100,82.03.10.2009,bottomup
14,"MENARA KARYA LT.22, JL. HR. RASUNA SAID BLK X-...",KALIMANTAN SELATAN,100,63,KABUPATEN BALANGAN,100,63.11,HALONG,100,63.11.02,KARYA,100,63.11.02.2018,bottomup


In [20]:
sample_df.to_excel("/Users/tiarasabrina/Documents/PROJECT/AI/address-parsing/master-data/master-matched/trial2_regex.xlsx", index=False)

##TRIAL 3

In [24]:
import re
import pandas as pd
from sqlalchemy import create_engine
from rapidfuzz import fuzz, process
from tqdm import tqdm

tqdm.pandas()

# ============================================
# 1. LOAD MASTER DATA WILAYAH DARI POSTGRES
# ============================================
engine = create_engine("postgresql://tiarasabrina@localhost:5432/wilayah_db")
wilayah = pd.read_sql("SELECT kode, nama FROM wilayah.wilayah", engine)
wilayah["nama"] = wilayah["nama"].str.upper().str.strip()
wilayah = wilayah.dropna(subset=["kode", "nama"])

def get_level(kode):
    parts = kode.split(".")
    return {1: "provinsi", 2: "kabkota", 3: "kecamatan", 4: "desa"}.get(len(parts), "unknown")

wilayah["level"] = wilayah["kode"].apply(get_level)

df_prov    = wilayah[wilayah["level"] == "provinsi"].reset_index(drop=True)
df_kabkota = wilayah[wilayah["level"] == "kabkota"].reset_index(drop=True)
df_kec     = wilayah[wilayah["level"] == "kecamatan"].reset_index(drop=True)
df_desa    = wilayah[wilayah["level"] == "desa"].reset_index(drop=True)

print(f"Provinsi: {len(df_prov)}, Kab/Kota: {len(df_kabkota)}, Kecamatan: {len(df_kec)}, Desa: {len(df_desa)}")
# WAJIB dicek sebelum lanjut, biar alias provinsi di bawah nyambung persis:
print(df_prov["nama"].tolist())


# ============================================
# 2. ALIAS PROVINSI (singkatan umum -> nama resmi PERSIS di df_prov)
# Sesuaikan lagi sisi kanan (value) kalau penulisan resminya beda dari yang dicetak di atas
# ============================================
PROVINCE_ALIASES = {
    "DKI": "DKI JAKARTA", "JAKARTA": "DKI JAKARTA",
    "JABAR": "JAWA BARAT", "JATENG": "JAWA TENGAH", "JATIM": "JAWA TIMUR",
    "KALSEL": "KALIMANTAN SELATAN", "KALTIM": "KALIMANTAN TIMUR",
    "KALBAR": "KALIMANTAN BARAT", "KALTENG": "KALIMANTAN TENGAH",
    "KALTARA": "KALIMANTAN UTARA",
    "SUMUT": "SUMATERA UTARA", "SUMBAR": "SUMATERA BARAT", "SUMSEL": "SUMATERA SELATAN",
    "SULSEL": "SULAWESI SELATAN", "SULUT": "SULAWESI UTARA", "SULTENG": "SULAWESI TENGAH",
    "SULTRA": "SULAWESI TENGGARA", "SULBAR": "SULAWESI BARAT",
    "NTB": "NUSA TENGGARA BARAT", "NTT": "NUSA TENGGARA TIMUR",
    "KEPRI": "KEPULAUAN RIAU", "BABEL": "KEPULAUAN BANGKA BELITUNG",
    "DIY": "DAERAH ISTIMEWA YOGYAKARTA", "YOGYA": "DAERAH ISTIMEWA YOGYAKARTA",
    "JOGJA": "DAERAH ISTIMEWA YOGYAKARTA",
    "PAPBAR": "PAPUA BARAT",
}


# ============================================
# 3. KATA KUNCI STRUKTURAL & PATTERN REGEX
# ============================================
BUILDING_KEYWORDS = r"\b(GEDUNG|GD|GDG|MENARA|TOWER|WISMA|GRAHA|GRIYA|PLAZA|MALL|RUKO|RUKAN|BUILDING|OFFICE|SUITE|KOMPLEK|KOMPLEKS|PERKANTORAN|APARTEMEN|APARTMENT|CENTRE|CENTER|SENTRA|PARK|SQUARE)\b"
STREET_KEYWORDS   = r"\b(JL|JLN|JALAN)\b"
FLOOR_PATTERN     = r"\b(LT|LANTAI)\.?\s*([A-Z0-9\-]+)\b"
NO_PATTERN        = r"\b(NO|NOMOR)\.?\s*([A-Z0-9\-\/]+)\b"
BLOK_PATTERN      = r"\bBLOK\.?\s*([A-Z0-9\-]+)\b"
KAV_PATTERN       = r"\bKAV(?:LING)?\.?\s*([A-Z0-9\-]+)\b"
POSTAL_PATTERN    = r"\b(\d{5})\b"
RT_RW_PATTERN     = r"\bRT\.?\s*/?\s*RW\.?\s*(\d+)\s*/\s*(\d+)\b|\bRT\.?\s*(\d+)\b.{0,10}\bRW\.?\s*(\d+)\b"


# ============================================
# 4. SEGMENTASI ALAMAT
# ============================================
def segment_address(alamat_upper):
    raw_segments = re.split(r"[,\n]", alamat_upper)
    raw_segments = [s.strip() for s in raw_segments if s.strip()]

    building_segments = []
    street_segments = []
    wilayah_candidate_segments = []

    for seg in raw_segments:
        if re.search(BUILDING_KEYWORDS, seg):
            building_segments.append(seg)
        elif re.search(STREET_KEYWORDS, seg):
            street_segments.append(seg)
        else:
            wilayah_candidate_segments.append(seg)

    return {
        "building": building_segments,
        "street": street_segments,
        "wilayah_candidates": wilayah_candidate_segments,
    }


# ============================================
# 5. EXTRACT NAMA JALAN DARI STREET SEGMENTS
# ============================================
def extract_street_name(street_segments):
    if not street_segments:
        return None

    cleaned_parts = []
    for seg in street_segments:
        s = seg
        s = re.sub(STREET_KEYWORDS, "", s)
        s = re.sub(NO_PATTERN, "", s)
        s = re.sub(FLOOR_PATTERN, "", s)
        s = re.sub(BLOK_PATTERN, "", s)
        s = re.sub(KAV_PATTERN, "", s)
        s = re.sub(RT_RW_PATTERN, "", s)
        s = re.sub(r"\s+", " ", s).strip(" .,-")
        if s:
            cleaned_parts.append(s)

    return " ".join(cleaned_parts) if cleaned_parts else None


# ============================================
# 6. EXACT MATCH DI DALAM SEGMEN WILAYAH (bukan seluruh string)
# ============================================
def exact_match_in_segments(segments_text, candidates_df, min_len=4):
    candidates_sorted = candidates_df.assign(
        _len=candidates_df["nama"].str.len()
    ).sort_values("_len", ascending=False)

    for _, row in candidates_sorted.iterrows():
        nama = row["nama"]
        if not nama or len(nama) < min_len:
            continue
        pattern = r"\b" + re.escape(nama) + r"\b"
        if re.search(pattern, segments_text):
            return nama, row["kode"]
    return None, None

def match_province_with_alias(segments_text):
    for alias, resmi in PROVINCE_ALIASES.items():
        if re.search(r"\b" + re.escape(alias) + r"\b", segments_text):
            row = df_prov[df_prov["nama"] == resmi]
            if not row.empty:
                return resmi, row.iloc[0]["kode"]
    return exact_match_in_segments(segments_text, df_prov, min_len=4)


# ============================================
# 7. MASTER FUNCTION
# ============================================
def match_address(alamat):
    result = {
        "jalan": None, "no": None, "lantai": None, "blok": None, "kav": None,
        "rt": None, "rw": None, "kodepos": None,
        "provinsi": None, "kode_prov": None,
        "kabkota": None, "kode_kabkota": None,
        "kecamatan": None, "kode_kec": None,
        "desa": None, "kode_desa": None,
        "remaining_text": None,
    }

    if not isinstance(alamat, str) or not alamat.strip() or alamat.strip().upper() == "NAN":
        return result

    alamat_upper = alamat.upper().replace("\n", " ")

    # --- field struktural ---
    m = re.search(NO_PATTERN, alamat_upper)
    if m: result["no"] = m.group(2)
    m = re.search(FLOOR_PATTERN, alamat_upper)
    if m: result["lantai"] = m.group(2)
    m = re.search(BLOK_PATTERN, alamat_upper)
    if m: result["blok"] = m.group(1)
    m = re.search(KAV_PATTERN, alamat_upper)
    if m: result["kav"] = m.group(1)
    m = re.search(POSTAL_PATTERN, alamat_upper)
    if m: result["kodepos"] = m.group(1)

    m = re.search(RT_RW_PATTERN, alamat_upper)
    if m:
        if m.group(1) and m.group(2):
            result["rt"], result["rw"] = m.group(1), m.group(2)
        elif m.group(3) and m.group(4):
            result["rt"], result["rw"] = m.group(3), m.group(4)

    # --- segmentasi ---
    seg = segment_address(alamat_upper)

    # --- nama jalan ---
    result["jalan"] = extract_street_name(seg["street"])

    # --- cascading match wilayah, TOP-DOWN, cuma di wilayah_candidates ---
    wilayah_text = " , ".join(seg["wilayah_candidates"])

    if wilayah_text:
        nama_prov, kode_prov = match_province_with_alias(wilayah_text)
        if nama_prov:
            result["provinsi"], result["kode_prov"] = nama_prov, kode_prov

            kandidat_kab = df_kabkota[df_kabkota["kode"].str.startswith(kode_prov + ".")]
            nama_kab, kode_kab = exact_match_in_segments(wilayah_text, kandidat_kab, min_len=4)
            if nama_kab:
                result["kabkota"], result["kode_kabkota"] = nama_kab, kode_kab

                kandidat_kec = df_kec[df_kec["kode"].str.startswith(kode_kab + ".")]
                nama_kec, kode_kec = exact_match_in_segments(wilayah_text, kandidat_kec, min_len=4)
                if nama_kec:
                    result["kecamatan"], result["kode_kec"] = nama_kec, kode_kec

                    kandidat_desa = df_desa[df_desa["kode"].str.startswith(kode_kec + ".")]
                    nama_desa, kode_desa = exact_match_in_segments(wilayah_text, kandidat_desa, min_len=6)
                    if nama_desa:
                        result["desa"], result["kode_desa"] = nama_desa, kode_desa

    # --- remaining_text: alamat asli dikurangi semua yang udah kekonsumsi field lain ---
    consumed = []
    for key in ["jalan", "provinsi", "kabkota", "kecamatan", "desa"]:
        if result[key]:
            consumed.append(result[key])
    for val in [result["no"], result["lantai"], result["blok"], result["kav"],
                result["kodepos"], result["rt"], result["rw"]]:
        if val:
            consumed.append(str(val))

    remaining = alamat_upper
    for c in consumed:
        remaining = remaining.replace(c, "")
    result["remaining_text"] = re.sub(r"\s+", " ", remaining).strip(" ,")

    return result


# ============================================
# 8. LOAD DATA ALAMAT & TEST KE SAMPLE
# ============================================
gold_company = pd.read_excel(
    "/Users/tiarasabrina/Documents/PROJECT/AI/address-parsing/master-data/master-matched/Master_Matching_20260701_1059 (1).xlsx",
    engine="openpyxl"
)
gold_company_address = gold_company[["Alamat Lengkap"]].copy()
gold_company_address["alamat_clean"] = (
    gold_company_address["Alamat Lengkap"].astype(str).str.upper().str.strip()
)

sample = gold_company_address["alamat_clean"].dropna().head(50)
sample_results = sample.progress_apply(match_address)
sample_df = pd.concat([sample, pd.DataFrame(sample_results.tolist(), index=sample.index)], axis=1)
sample_df

Provinsi: 38, Kab/Kota: 514, Kecamatan: 7285, Desa: 83762
['ACEH', 'SUMATERA UTARA', 'SUMATERA BARAT', 'RIAU', 'JAMBI', 'SUMATERA SELATAN', 'BENGKULU', 'LAMPUNG', 'KEPULAUAN BANGKA BELITUNG', 'KEPULAUAN RIAU', 'DAERAH KHUSUS IBUKOTA JAKARTA', 'JAWA BARAT', 'JAWA TENGAH', 'DAERAH ISTIMEWA YOGYAKARTA', 'JAWA TIMUR', 'BANTEN', 'BALI', 'NUSA TENGGARA BARAT', 'NUSA TENGGARA TIMUR', 'KALIMANTAN BARAT', 'KALIMANTAN TENGAH', 'KALIMANTAN SELATAN', 'KALIMANTAN TIMUR', 'KALIMANTAN UTARA', 'SULAWESI UTARA', 'SULAWESI TENGAH', 'SULAWESI SELATAN', 'SULAWESI TENGGARA', 'GORONTALO', 'SULAWESI BARAT', 'MALUKU', 'MALUKU UTARA', 'PAPUA', 'PAPUA BARAT', 'PAPUA SELATAN', 'PAPUA TENGAH', 'PAPUA PEGUNUNGAN', 'PAPUA BARAT DAYA']


100%|██████████| 50/50 [00:00<00:00, 359.49it/s]


,alamat_clean,jalan,no,lantai,blok,kav,rt,rw,kodepos,provinsi,kode_prov,kabkota,kode_kabkota,kecamatan,kode_kec,desa,kode_desa,remaining_text
0,CYBER 2 TOWER LANTAI 26\nJL. H.R. RASUNA SAID ...,H.R. RASUNA SAID,13,26,X-5,NaN,NaN,NaN,12950,NaN,NaN,None,None,None,None,None,None,"CYBER 2 TOWER LANTAI JL. BLOK , NO. JAKARTA - ..."
6,JL. GURAMI NO. 173,GURAMI,173,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,None,JL. NO.
7,"MENARA KARYA LANTAI 22, JL. H.R. RASUNA SAID B...",H.R. RASUNA SAID,NaN,22,X-5,1-2,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,None,"MENARA KARYA LANTAI , JL. BLOK KAV."
8,"GEDUNG CYBER 2 TOWER LT.23, JL. HR RASUNA SAID...",HR RASUNA SAID,13,23,X-5,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,None,"GEDUNG CYBER 2 TOWER LT., JL. BLOK , NO."
9,"GD. MENARA KARYA LT.23, JL.HR. RASUNA SAID BLO...",HR. RASUNA SAID,NaN,23,X-5,1-2,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,None,"GD. MENARA KARYA LT., JL. BLOK KAV."
10,MENARA KARYA LT.23 JL.HR.RASUNA SAID BLOK X-5 ...,NaN,NaN,23,X-5,1-2,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,None,MENARA KARYA LT. JL.HR.RASUNA SAID BLOK KAV.
11,"GEDUNG MENARA KARYA LT. 18, JL. H.R. RASUNA SA...",H.R. RASUNA SAID,NaN,18,X-5,1-2,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,None,"GEDUNG MENARA KARYA LT. , JL. , BLOK , KAV."
12,JL. KOM. LAUT. YOS SUDARSO (DALAM GANG PRIBADI...,KOM. LAUT. YOS SUDARSO (DALAM GANG PRIBADI LIN...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,None,JL.
13,RUKO GREEN GARDEN BLOK A2-10 JL. WAHIDIN SUDIR...,NaN,NaN,NaN,A2-10,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,None,RUKO GREEN GARDEN BLOK JL. WAHIDIN SUDIROHUSODO
14,"MENARA KARYA LT.22, JL. HR. RASUNA SAID BLK X-...",HR. RASUNA SAID BLK X-5,NaN,22,NaN,1-2,NaN,NaN,NaN,NaN,NaN,None,None,None,None,None,None,"MENARA KARYA LT., JL. , KAV."


In [25]:
#trial 3 save
sample_df.to_excel("/Users/tiarasabrina/Documents/PROJECT/AI/address-parsing/master-data/master-matched/trial3_regex.xlsx", index=False)

##TRIAL 4

In [26]:
import re
import pandas as pd
from sqlalchemy import create_engine
from tqdm import tqdm

tqdm.pandas()

# ============================================
# 1. LOAD MASTER DATA WILAYAH DARI POSTGRES
# ============================================
engine = create_engine("postgresql://tiarasabrina@localhost:5432/wilayah_db")
wilayah = pd.read_sql("SELECT kode, nama FROM wilayah.wilayah", engine)
wilayah["nama"] = wilayah["nama"].str.upper().str.strip()
wilayah = wilayah.dropna(subset=["kode", "nama"])

def get_level(kode):
    parts = kode.split(".")
    return {1: "provinsi", 2: "kabkota", 3: "kecamatan", 4: "desa"}.get(len(parts), "unknown")

wilayah["level"] = wilayah["kode"].apply(get_level)

df_prov    = wilayah[wilayah["level"] == "provinsi"].reset_index(drop=True)
df_kabkota = wilayah[wilayah["level"] == "kabkota"].reset_index(drop=True)
df_kec     = wilayah[wilayah["level"] == "kecamatan"].reset_index(drop=True)
df_desa    = wilayah[wilayah["level"] == "desa"].reset_index(drop=True)

print(f"Provinsi: {len(df_prov)}, Kab/Kota: {len(df_kabkota)}, Kecamatan: {len(df_kec)}, Desa: {len(df_desa)}")
print(df_prov["nama"].tolist())  # cek dulu penulisan resmi biar alias di bawah nyambung


# ============================================
# 2. ALIAS PROVINSI (sesuaikan sisi kanan kalau beda dari hasil print di atas)
# ============================================
PROVINCE_ALIASES = {
    "DKI": "DKI JAKARTA", "JAKARTA": "DKI JAKARTA",
    "JABAR": "JAWA BARAT", "JATENG": "JAWA TENGAH", "JATIM": "JAWA TIMUR",
    "KALSEL": "KALIMANTAN SELATAN", "KALTIM": "KALIMANTAN TIMUR",
    "KALBAR": "KALIMANTAN BARAT", "KALTENG": "KALIMANTAN TENGAH",
    "KALTARA": "KALIMANTAN UTARA",
    "SUMUT": "SUMATERA UTARA", "SUMBAR": "SUMATERA BARAT", "SUMSEL": "SUMATERA SELATAN",
    "SULSEL": "SULAWESI SELATAN", "SULUT": "SULAWESI UTARA", "SULTENG": "SULAWESI TENGAH",
    "SULTRA": "SULAWESI TENGGARA", "SULBAR": "SULAWESI BARAT",
    "NTB": "NUSA TENGGARA BARAT", "NTT": "NUSA TENGGARA TIMUR",
    "KEPRI": "KEPULAUAN RIAU", "BABEL": "KEPULAUAN BANGKA BELITUNG",
    "DIY": "DAERAH ISTIMEWA YOGYAKARTA", "YOGYA": "DAERAH ISTIMEWA YOGYAKARTA",
    "JOGJA": "DAERAH ISTIMEWA YOGYAKARTA",
    "PAPBAR": "PAPUA BARAT",
}


# ============================================
# 3. KATA KUNCI STRUKTURAL & PATTERN REGEX
# ============================================
BUILDING_KEYWORDS = r"\b(GEDUNG|GD|GDG|MENARA|TOWER|WISMA|GRAHA|GRIYA|PLAZA|MALL|RUKO|RUKAN|BUILDING|OFFICE|SUITE|KOMPLEK|KOMPLEKS|PERKANTORAN|APARTEMEN|APARTMENT|CENTRE|CENTER|SENTRA|PARK|SQUARE)\b"
STREET_KEYWORDS   = r"\b(JL|JLN|JALAN)\b"
FLOOR_PATTERN     = r"\b(LT|LANTAI)\.?\s*([A-Z0-9\-]+)\b"
NO_PATTERN        = r"\b(NO|NOMOR)\.?\s*([A-Z0-9\-\/]+)\b"
BLOK_PATTERN      = r"\bBLOK\.?\s*([A-Z0-9\-]+)\b"
KAV_PATTERN       = r"\bKAV(?:LING)?\.?\s*([A-Z0-9\-]+)\b"
POSTAL_PATTERN    = r"\b(\d{5})\b"
RT_RW_PATTERN     = r"\bRT\.?\s*/?\s*RW\.?\s*(\d+)\s*/\s*(\d+)\b|\bRT\.?\s*(\d+)\b.{0,10}\bRW\.?\s*(\d+)\b"


# ============================================
# 4. SEGMENTASI ALAMAT
# ============================================
def segment_address(alamat_upper):
    raw_segments = re.split(r"[,\n]", alamat_upper)
    raw_segments = [s.strip() for s in raw_segments if s.strip()]

    building_segments = []
    street_segments = []
    wilayah_candidate_segments = []

    for seg in raw_segments:
        if re.search(BUILDING_KEYWORDS, seg):
            building_segments.append(seg)
        elif re.search(STREET_KEYWORDS, seg):
            street_segments.append(seg)
        else:
            wilayah_candidate_segments.append(seg)

    return {
        "building": building_segments,
        "street": street_segments,
        "wilayah_candidates": wilayah_candidate_segments,
    }


# ============================================
# 5. EXTRACT NAMA GEDUNG & NAMA JALAN (jadi kolom terpisah)
# ============================================
def clean_segment_generic(s):
    """Buang NO/LANTAI/BLOK/KAV/RT-RW yang nempel di segmen (udah ditangkep field lain)."""
    s = re.sub(NO_PATTERN, "", s)
    s = re.sub(FLOOR_PATTERN, "", s)
    s = re.sub(BLOK_PATTERN, "", s)
    s = re.sub(KAV_PATTERN, "", s)
    s = re.sub(RT_RW_PATTERN, "", s)
    s = re.sub(r"\s+", " ", s).strip(" .,-")
    return s

def extract_building_name(building_segments):
    if not building_segments:
        return None
    cleaned_parts = [clean_segment_generic(seg) for seg in building_segments]
    cleaned_parts = [c for c in cleaned_parts if c]
    return " ".join(cleaned_parts) if cleaned_parts else None

def extract_street_name(street_segments):
    if not street_segments:
        return None
    cleaned_parts = []
    for seg in street_segments:
        s = re.sub(STREET_KEYWORDS, "", seg)
        s = clean_segment_generic(s)
        if s:
            cleaned_parts.append(s)
    return " ".join(cleaned_parts) if cleaned_parts else None


# ============================================
# 6. EXACT MATCH (toleran spasi: "SETIA BUDI" == "SETIABUDI")
# ============================================
def normalize_for_match(text):
    return re.sub(r"\s+", "", text)

def exact_match_in_segments(segments_text, candidates_df, min_len=4):
    segments_nospace = normalize_for_match(segments_text)
    candidates_sorted = candidates_df.assign(
        _len=candidates_df["nama"].str.len()
    ).sort_values("_len", ascending=False)

    for _, row in candidates_sorted.iterrows():
        nama = row["nama"]
        if not nama or len(nama) < min_len:
            continue
        nama_nospace = normalize_for_match(nama)

        # coba exact word-boundary di teks asli (dengan spasi)
        pattern = r"\b" + re.escape(nama) + r"\b"
        if re.search(pattern, segments_text):
            return nama, row["kode"]

        # coba versi tanpa spasi di kandidat vs teks asli (misal "SETIABUDI" vs "SETIA BUDI")
        pattern_nospace_candidate = r"\b" + re.escape(nama.replace(" ", "")) + r"\b"
        if re.search(pattern_nospace_candidate, segments_text):
            return nama, row["kode"]

        # coba versi tanpa spasi keduanya (paling longgar, masih exact substring)
        if nama_nospace in segments_nospace and len(nama_nospace) >= min_len:
            return nama, row["kode"]

    return None, None

def match_province_with_alias(segments_text):
    for alias, resmi in PROVINCE_ALIASES.items():
        if re.search(r"\b" + re.escape(alias) + r"\b", segments_text):
            row = df_prov[df_prov["nama"] == resmi]
            if not row.empty:
                return resmi, row.iloc[0]["kode"]
    return exact_match_in_segments(segments_text, df_prov, min_len=4)


# ============================================
# 7. MATCH WILAYAH LANGSUNG (desa -> kecamatan -> kabkota -> provinsi)
# tanpa syarat harus ketemu provinsi dulu; level atas di-derive dari kode
# ============================================
def _fill_upper_levels(result, kode_kec, kode_kab, kode_prov):
    if kode_kec and not result["kecamatan"]:
        row = df_kec[df_kec["kode"] == kode_kec]
        if not row.empty:
            result["kecamatan"], result["kode_kec"] = row.iloc[0]["nama"], kode_kec
    if kode_kab and not result["kabkota"]:
        row = df_kabkota[df_kabkota["kode"] == kode_kab]
        if not row.empty:
            result["kabkota"], result["kode_kabkota"] = row.iloc[0]["nama"], kode_kab
    if kode_prov and not result["provinsi"]:
        row = df_prov[df_prov["kode"] == kode_prov]
        if not row.empty:
            result["provinsi"], result["kode_prov"] = row.iloc[0]["nama"], kode_prov

def match_wilayah_direct(wilayah_text):
    result = {
        "provinsi": None, "kode_prov": None,
        "kabkota": None, "kode_kabkota": None,
        "kecamatan": None, "kode_kec": None,
        "desa": None, "kode_desa": None,
    }

    # 1. desa dulu (paling spesifik)
    nama, kode = exact_match_in_segments(wilayah_text, df_desa, min_len=6)
    if nama:
        result["desa"], result["kode_desa"] = nama, kode
        parts = kode.split(".")
        kode_kec, kode_kab, kode_prov = ".".join(parts[:3]), ".".join(parts[:2]), parts[0]
        _fill_upper_levels(result, kode_kec, kode_kab, kode_prov)
        return result

    # 2. kecamatan
    nama, kode = exact_match_in_segments(wilayah_text, df_kec, min_len=5)
    if nama:
        result["kecamatan"], result["kode_kec"] = nama, kode
        parts = kode.split(".")
        kode_kab, kode_prov = ".".join(parts[:2]), parts[0]
        _fill_upper_levels(result, None, kode_kab, kode_prov)
        return result

    # 3. kabkota
    nama, kode = exact_match_in_segments(wilayah_text, df_kabkota, min_len=4)
    if nama:
        result["kabkota"], result["kode_kabkota"] = nama, kode
        kode_prov = kode.split(".")[0]
        _fill_upper_levels(result, None, None, kode_prov)
        return result

    # 4. provinsi (pakai alias)
    nama, kode = match_province_with_alias(wilayah_text)
    if nama:
        result["provinsi"], result["kode_prov"] = nama, kode

    return result


# ============================================
# 8. MASTER FUNCTION
# ============================================
def match_address(alamat):
    result = {
        "gedung": None, "jalan": None,
        "no": None, "lantai": None, "blok": None, "kav": None,
        "rt": None, "rw": None, "kodepos": None,
        "provinsi": None, "kode_prov": None,
        "kabkota": None, "kode_kabkota": None,
        "kecamatan": None, "kode_kec": None,
        "desa": None, "kode_desa": None,
        "remaining_text": None,
    }

    if not isinstance(alamat, str) or not alamat.strip() or alamat.strip().upper() == "NAN":
        return result

    alamat_upper = alamat.upper().replace("\n", " ")

    # --- field struktural ---
    m = re.search(NO_PATTERN, alamat_upper)
    if m: result["no"] = m.group(2)
    m = re.search(FLOOR_PATTERN, alamat_upper)
    if m: result["lantai"] = m.group(2)
    m = re.search(BLOK_PATTERN, alamat_upper)
    if m: result["blok"] = m.group(1)
    m = re.search(KAV_PATTERN, alamat_upper)
    if m: result["kav"] = m.group(1)
    m = re.search(POSTAL_PATTERN, alamat_upper)
    if m: result["kodepos"] = m.group(1)

    m = re.search(RT_RW_PATTERN, alamat_upper)
    if m:
        if m.group(1) and m.group(2):
            result["rt"], result["rw"] = m.group(1), m.group(2)
        elif m.group(3) and m.group(4):
            result["rt"], result["rw"] = m.group(3), m.group(4)

    # --- segmentasi ---
    seg = segment_address(alamat_upper)

    # --- gedung & jalan jadi kolom sendiri ---
    result["gedung"] = extract_building_name(seg["building"])
    result["jalan"] = extract_street_name(seg["street"])

    # --- match wilayah langsung, level manapun yang ketemu duluan ---
    wilayah_text = " , ".join(seg["wilayah_candidates"])
    if wilayah_text:
        wilayah_result = match_wilayah_direct(wilayah_text)
        result.update(wilayah_result)

    # --- remaining_text: sisa yang belum kekonsumsi field manapun ---
    consumed = []
    for key in ["gedung", "jalan", "provinsi", "kabkota", "kecamatan", "desa"]:
        if result[key]:
            consumed.append(result[key])
    for val in [result["no"], result["lantai"], result["blok"], result["kav"],
                result["kodepos"], result["rt"], result["rw"]]:
        if val:
            consumed.append(str(val))

    remaining = alamat_upper
    for c in consumed:
        remaining = remaining.replace(c, "")
    result["remaining_text"] = re.sub(r"\s+", " ", remaining).strip(" ,")

    return result


# ============================================
# 9. LOAD DATA ALAMAT & TEST KE SAMPLE
# ============================================
gold_company = pd.read_excel(
    "/Users/tiarasabrina/Documents/PROJECT/AI/address-parsing/master-data/master-matched/Master_Matching_20260701_1059 (1).xlsx",
    engine="openpyxl"
)
gold_company_address = gold_company[["Alamat Lengkap"]].copy()
gold_company_address["alamat_clean"] = (
    gold_company_address["Alamat Lengkap"].astype(str).str.upper().str.strip()
)

sample = gold_company_address["alamat_clean"].dropna().head(50)
sample_results = sample.progress_apply(match_address)
sample_df = pd.concat([sample, pd.DataFrame(sample_results.tolist(), index=sample.index)], axis=1)
sample_df

Provinsi: 38, Kab/Kota: 514, Kecamatan: 7285, Desa: 83762
['ACEH', 'SUMATERA UTARA', 'SUMATERA BARAT', 'RIAU', 'JAMBI', 'SUMATERA SELATAN', 'BENGKULU', 'LAMPUNG', 'KEPULAUAN BANGKA BELITUNG', 'KEPULAUAN RIAU', 'DAERAH KHUSUS IBUKOTA JAKARTA', 'JAWA BARAT', 'JAWA TENGAH', 'DAERAH ISTIMEWA YOGYAKARTA', 'JAWA TIMUR', 'BANTEN', 'BALI', 'NUSA TENGGARA BARAT', 'NUSA TENGGARA TIMUR', 'KALIMANTAN BARAT', 'KALIMANTAN TENGAH', 'KALIMANTAN SELATAN', 'KALIMANTAN TIMUR', 'KALIMANTAN UTARA', 'SULAWESI UTARA', 'SULAWESI TENGAH', 'SULAWESI SELATAN', 'SULAWESI TENGGARA', 'GORONTALO', 'SULAWESI BARAT', 'MALUKU', 'MALUKU UTARA', 'PAPUA', 'PAPUA BARAT', 'PAPUA SELATAN', 'PAPUA TENGAH', 'PAPUA PEGUNUNGAN', 'PAPUA BARAT DAYA']


100%|██████████| 50/50 [01:17<00:00,  1.56s/it]


,alamat_clean,gedung,jalan,no,lantai,blok,kav,rt,rw,kodepos,provinsi,kode_prov,kabkota,kode_kabkota,kecamatan,kode_kec,desa,kode_desa,remaining_text
0,CYBER 2 TOWER LANTAI 26\nJL. H.R. RASUNA SAID ...,CYBER 2 TOWER JL. H.R. RASUNA SAID JAKARTA 129...,H.R. RASUNA SAID,13,26,X-5,NaN,NaN,NaN,12950,BALI,51,KABUPATEN KARANGASEM,51.07,SELAT,51.07.07,NaN,NaN,"CYBER 2 TOWER LANTAI JL. BLOK , NO. JAKARTA - ..."
6,JL. GURAMI NO. 173,NaN,GURAMI,173,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,JL. NO.
7,"MENARA KARYA LANTAI 22, JL. H.R. RASUNA SAID B...",MENARA KARYA,H.R. RASUNA SAID,NaN,22,X-5,1-2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LANTAI , JL. BLOK KAV."
8,"GEDUNG CYBER 2 TOWER LT.23, JL. HR RASUNA SAID...",GEDUNG CYBER 2 TOWER,HR RASUNA SAID,13,23,X-5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LT., JL. BLOK , NO."
9,"GD. MENARA KARYA LT.23, JL.HR. RASUNA SAID BLO...",GD. MENARA KARYA,HR. RASUNA SAID,NaN,23,X-5,1-2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LT., JL. BLOK KAV."
10,MENARA KARYA LT.23 JL.HR.RASUNA SAID BLOK X-5 ...,MENARA KARYA JL.HR.RASUNA SAID,NaN,NaN,23,X-5,1-2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MENARA KARYA LT. JL.HR.RASUNA SAID BLOK KAV.
11,"GEDUNG MENARA KARYA LT. 18, JL. H.R. RASUNA SA...",GEDUNG MENARA KARYA,H.R. RASUNA SAID,NaN,18,X-5,1-2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LT. , JL. , BLOK , KAV."
12,JL. KOM. LAUT. YOS SUDARSO (DALAM GANG PRIBADI...,NaN,KOM. LAUT. YOS SUDARSO (DALAM GANG PRIBADI LIN...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,JL.
13,RUKO GREEN GARDEN BLOK A2-10 JL. WAHIDIN SUDIR...,RUKO GREEN GARDEN JL. WAHIDIN SUDIROHUSODO,NaN,NaN,NaN,A2-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RUKO GREEN GARDEN BLOK JL. WAHIDIN SUDIROHUSODO
14,"MENARA KARYA LT.22, JL. HR. RASUNA SAID BLK X-...",MENARA KARYA,HR. RASUNA SAID BLK X-5,NaN,22,NaN,1-2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LT., JL. , KAV."


In [27]:
## trial 4 save
sample_df.to_excel("/Users/tiarasabrina/Documents/PROJECT/AI/address-parsing/master-data/master-matched/trial4_regex.xlsx", index=False)

## TRIAL 5

In [28]:
import re
import pandas as pd
from sqlalchemy import create_engine
from tqdm import tqdm

tqdm.pandas()

# ============================================
# 1. LOAD MASTER DATA WILAYAH DARI POSTGRES
# ============================================
engine = create_engine("postgresql://tiarasabrina@localhost:5432/wilayah_db")
wilayah = pd.read_sql("SELECT kode, nama FROM wilayah.wilayah", engine)
wilayah["nama"] = wilayah["nama"].str.upper().str.strip()
wilayah = wilayah.dropna(subset=["kode", "nama"])

def get_level(kode):
    parts = kode.split(".")
    return {1: "provinsi", 2: "kabkota", 3: "kecamatan", 4: "desa"}.get(len(parts), "unknown")

wilayah["level"] = wilayah["kode"].apply(get_level)

df_prov    = wilayah[wilayah["level"] == "provinsi"].reset_index(drop=True)
df_kabkota = wilayah[wilayah["level"] == "kabkota"].reset_index(drop=True)
df_kec     = wilayah[wilayah["level"] == "kecamatan"].reset_index(drop=True)
df_desa    = wilayah[wilayah["level"] == "desa"].reset_index(drop=True)

print(f"Provinsi: {len(df_prov)}, Kab/Kota: {len(df_kabkota)}, Kecamatan: {len(df_kec)}, Desa: {len(df_desa)}")
print(df_prov["nama"].tolist())  # cek dulu penulisan resmi biar alias di bawah nyambung


# ============================================
# 2. ALIAS PROVINSI (sesuaikan sisi kanan kalau beda dari hasil print di atas)
# ============================================
PROVINCE_ALIASES = {
    "DKI": "DKI JAKARTA", "JAKARTA": "DKI JAKARTA",
    "JABAR": "JAWA BARAT", "JATENG": "JAWA TENGAH", "JATIM": "JAWA TIMUR",
    "KALSEL": "KALIMANTAN SELATAN", "KALTIM": "KALIMANTAN TIMUR",
    "KALBAR": "KALIMANTAN BARAT", "KALTENG": "KALIMANTAN TENGAH",
    "KALTARA": "KALIMANTAN UTARA",
    "SUMUT": "SUMATERA UTARA", "SUMBAR": "SUMATERA BARAT", "SUMSEL": "SUMATERA SELATAN",
    "SULSEL": "SULAWESI SELATAN", "SULUT": "SULAWESI UTARA", "SULTENG": "SULAWESI TENGAH",
    "SULTRA": "SULAWESI TENGGARA", "SULBAR": "SULAWESI BARAT",
    "NTB": "NUSA TENGGARA BARAT", "NTT": "NUSA TENGGARA TIMUR",
    "KEPRI": "KEPULAUAN RIAU", "BABEL": "KEPULAUAN BANGKA BELITUNG",
    "DIY": "DAERAH ISTIMEWA YOGYAKARTA", "YOGYA": "DAERAH ISTIMEWA YOGYAKARTA",
    "JOGJA": "DAERAH ISTIMEWA YOGYAKARTA",
    "PAPBAR": "PAPUA BARAT",
}


# ============================================
# 3. KATA KUNCI STRUKTURAL & PATTERN REGEX
# ============================================
BUILDING_KEYWORDS = r"\b(GEDUNG|GD|GDG|MENARA|TOWER|WISMA|GRAHA|GRIYA|PLAZA|MALL|RUKO|RUKAN|BUILDING|OFFICE|SUITE|KOMPLEK|KOMPLEKS|PERKANTORAN|APARTEMEN|APARTMENT|CENTRE|CENTER|SENTRA|PARK|SQUARE)\b"
STREET_KEYWORDS   = r"\b(JL|JLN|JALAN)\b"
FLOOR_PATTERN     = r"\b(LT|LANTAI)\.?\s*([A-Z0-9\-]+)\b"
NO_PATTERN        = r"\b(NO|NOMOR)\.?\s*([A-Z0-9\-\/]+)\b"
BLOK_PATTERN      = r"\bBLOK\.?\s*([A-Z0-9\-]+)\b"
KAV_PATTERN       = r"\bKAV(?:LING)?\.?\s*([A-Z0-9\-]+)\b"
POSTAL_PATTERN    = r"\b(\d{5})\b"
RT_RW_PATTERN     = r"\bRT\.?\s*/?\s*RW\.?\s*(\d+)\s*/\s*(\d+)\b|\bRT\.?\s*(\d+)\b.{0,10}\bRW\.?\s*(\d+)\b"


# ============================================
# 4. SEGMENTASI ALAMAT
# ============================================
def segment_address(alamat_upper):
    raw_segments = re.split(r"[,\n]", alamat_upper)
    raw_segments = [s.strip() for s in raw_segments if s.strip()]

    building_segments = []
    street_segments = []
    wilayah_candidate_segments = []

    for seg in raw_segments:
        if re.search(BUILDING_KEYWORDS, seg):
            building_segments.append(seg)
        elif re.search(STREET_KEYWORDS, seg):
            street_segments.append(seg)
        else:
            wilayah_candidate_segments.append(seg)

    return {
        "building": building_segments,
        "street": street_segments,
        "wilayah_candidates": wilayah_candidate_segments,
    }


# ============================================
# 5. EXTRACT NAMA GEDUNG & NAMA JALAN (jadi kolom terpisah)
# ============================================
def clean_segment_generic(s):
    """Buang NO/LANTAI/BLOK/KAV/RT-RW yang nempel di segmen (udah ditangkep field lain)."""
    s = re.sub(NO_PATTERN, "", s)
    s = re.sub(FLOOR_PATTERN, "", s)
    s = re.sub(BLOK_PATTERN, "", s)
    s = re.sub(KAV_PATTERN, "", s)
    s = re.sub(RT_RW_PATTERN, "", s)
    s = re.sub(r"\s+", " ", s).strip(" .,-")
    return s

def extract_building_name(building_segments):
    if not building_segments:
        return None
    cleaned_parts = [clean_segment_generic(seg) for seg in building_segments]
    cleaned_parts = [c for c in cleaned_parts if c]
    return " ".join(cleaned_parts) if cleaned_parts else None

def extract_street_name(street_segments):
    if not street_segments:
        return None
    cleaned_parts = []
    for seg in street_segments:
        s = re.sub(STREET_KEYWORDS, "", seg)
        s = clean_segment_generic(s)
        if s:
            cleaned_parts.append(s)
    return " ".join(cleaned_parts) if cleaned_parts else None


# ============================================
# 6. EXACT MATCH (toleran spasi: "SETIA BUDI" == "SETIABUDI")
# ============================================
def normalize_for_match(text):
    return re.sub(r"\s+", "", text)

def exact_match_in_segments(segments_text, candidates_df, min_len=4):
    segments_nospace = normalize_for_match(segments_text)
    candidates_sorted = candidates_df.assign(
        _len=candidates_df["nama"].str.len()
    ).sort_values("_len", ascending=False)

    for _, row in candidates_sorted.iterrows():
        nama = row["nama"]
        if not nama or len(nama) < min_len:
            continue
        nama_nospace = normalize_for_match(nama)

        # coba exact word-boundary di teks asli (dengan spasi)
        pattern = r"\b" + re.escape(nama) + r"\b"
        if re.search(pattern, segments_text):
            return nama, row["kode"]

        # coba versi tanpa spasi di kandidat vs teks asli (misal "SETIABUDI" vs "SETIA BUDI")
        pattern_nospace_candidate = r"\b" + re.escape(nama.replace(" ", "")) + r"\b"
        if re.search(pattern_nospace_candidate, segments_text):
            return nama, row["kode"]

        # coba versi tanpa spasi keduanya (paling longgar, masih exact substring)
        if nama_nospace in segments_nospace and len(nama_nospace) >= min_len:
            return nama, row["kode"]

    return None, None

def match_province_with_alias(segments_text):
    for alias, resmi in PROVINCE_ALIASES.items():
        if re.search(r"\b" + re.escape(alias) + r"\b", segments_text):
            row = df_prov[df_prov["nama"] == resmi]
            if not row.empty:
                return resmi, row.iloc[0]["kode"]
    return exact_match_in_segments(segments_text, df_prov, min_len=4)


# ============================================
# 7. MATCH WILAYAH LANGSUNG (desa -> kecamatan -> kabkota -> provinsi)
# tanpa syarat harus ketemu provinsi dulu; level atas di-derive dari kode
# ============================================
def _fill_upper_levels(result, kode_kec, kode_kab, kode_prov):
    if kode_kec and not result["kecamatan"]:
        row = df_kec[df_kec["kode"] == kode_kec]
        if not row.empty:
            result["kecamatan"], result["kode_kec"] = row.iloc[0]["nama"], kode_kec
    if kode_kab and not result["kabkota"]:
        row = df_kabkota[df_kabkota["kode"] == kode_kab]
        if not row.empty:
            result["kabkota"], result["kode_kabkota"] = row.iloc[0]["nama"], kode_kab
    if kode_prov and not result["provinsi"]:
        row = df_prov[df_prov["kode"] == kode_prov]
        if not row.empty:
            result["provinsi"], result["kode_prov"] = row.iloc[0]["nama"], kode_prov

def match_wilayah_direct(wilayah_text):
    result = {
        "provinsi": None, "kode_prov": None,
        "kabkota": None, "kode_kabkota": None,
        "kecamatan": None, "kode_kec": None,
        "desa": None, "kode_desa": None,
    }

    # 1. desa dulu (paling spesifik)
    nama, kode = exact_match_in_segments(wilayah_text, df_desa, min_len=6)
    if nama:
        result["desa"], result["kode_desa"] = nama, kode
        parts = kode.split(".")
        kode_kec, kode_kab, kode_prov = ".".join(parts[:3]), ".".join(parts[:2]), parts[0]
        _fill_upper_levels(result, kode_kec, kode_kab, kode_prov)
        return result

    # 2. kecamatan
    nama, kode = exact_match_in_segments(wilayah_text, df_kec, min_len=5)
    if nama:
        result["kecamatan"], result["kode_kec"] = nama, kode
        parts = kode.split(".")
        kode_kab, kode_prov = ".".join(parts[:2]), parts[0]
        _fill_upper_levels(result, None, kode_kab, kode_prov)
        return result

    # 3. kabkota
    nama, kode = exact_match_in_segments(wilayah_text, df_kabkota, min_len=4)
    if nama:
        result["kabkota"], result["kode_kabkota"] = nama, kode
        kode_prov = kode.split(".")[0]
        _fill_upper_levels(result, None, None, kode_prov)
        return result

    # 4. provinsi (pakai alias)
    nama, kode = match_province_with_alias(wilayah_text)
    if nama:
        result["provinsi"], result["kode_prov"] = nama, kode

    return result


# ============================================
# 8. MASTER FUNCTION
# ============================================
def match_address(alamat):
    result = {
        "gedung": None, "jalan": None,
        "no": None, "lantai": None, "blok": None, "kav": None,
        "rt": None, "rw": None, "kodepos": None,
        "provinsi": None, "kode_prov": None,
        "kabkota": None, "kode_kabkota": None,
        "kecamatan": None, "kode_kec": None,
        "desa": None, "kode_desa": None,
        "remaining_text": None,
    }

    if not isinstance(alamat, str) or not alamat.strip() or alamat.strip().upper() == "NAN":
        return result

    alamat_upper = alamat.upper().replace("\n", " ")

    # --- field struktural ---
    m = re.search(NO_PATTERN, alamat_upper)
    if m: result["no"] = m.group(2)
    m = re.search(FLOOR_PATTERN, alamat_upper)
    if m: result["lantai"] = m.group(2)
    m = re.search(BLOK_PATTERN, alamat_upper)
    if m: result["blok"] = m.group(1)
    m = re.search(KAV_PATTERN, alamat_upper)
    if m: result["kav"] = m.group(1)
    m = re.search(POSTAL_PATTERN, alamat_upper)
    if m: result["kodepos"] = m.group(1)

    m = re.search(RT_RW_PATTERN, alamat_upper)
    if m:
        if m.group(1) and m.group(2):
            result["rt"], result["rw"] = m.group(1), m.group(2)
        elif m.group(3) and m.group(4):
            result["rt"], result["rw"] = m.group(3), m.group(4)

    # --- segmentasi ---
    seg = segment_address(alamat_upper)

    # --- gedung & jalan jadi kolom sendiri ---
    result["gedung"] = extract_building_name(seg["building"])
    result["jalan"] = extract_street_name(seg["street"])

    # --- match wilayah langsung, level manapun yang ketemu duluan ---
    wilayah_text = " , ".join(seg["wilayah_candidates"])
    if wilayah_text:
        wilayah_result = match_wilayah_direct(wilayah_text)
        result.update(wilayah_result)

    # --- remaining_text: sisa yang belum kekonsumsi field manapun ---
    consumed = []
    for key in ["gedung", "jalan", "provinsi", "kabkota", "kecamatan", "desa"]:
        if result[key]:
            consumed.append(result[key])
    for val in [result["no"], result["lantai"], result["blok"], result["kav"],
                result["kodepos"], result["rt"], result["rw"]]:
        if val:
            consumed.append(str(val))

    remaining = alamat_upper
    for c in consumed:
        remaining = remaining.replace(c, "")
    result["remaining_text"] = re.sub(r"\s+", " ", remaining).strip(" ,")

    return result


# ============================================
# 9. LOAD DATA ALAMAT & TEST KE SAMPLE
# ============================================
gold_company = pd.read_excel(
    "/Users/tiarasabrina/Documents/PROJECT/AI/address-parsing/master-data/master-matched/Master_Matching_20260701_1059 (1).xlsx",
    engine="openpyxl"
)
gold_company_address = gold_company[["Alamat Lengkap"]].copy()
gold_company_address["alamat_clean"] = (
    gold_company_address["Alamat Lengkap"].astype(str).str.upper().str.strip()
)

sample = gold_company_address["alamat_clean"].dropna().head(50)
sample_results = sample.progress_apply(match_address)
sample_df = pd.concat([sample, pd.DataFrame(sample_results.tolist(), index=sample.index)], axis=1)
sample_df

Provinsi: 38, Kab/Kota: 514, Kecamatan: 7285, Desa: 83762
['ACEH', 'SUMATERA UTARA', 'SUMATERA BARAT', 'RIAU', 'JAMBI', 'SUMATERA SELATAN', 'BENGKULU', 'LAMPUNG', 'KEPULAUAN BANGKA BELITUNG', 'KEPULAUAN RIAU', 'DAERAH KHUSUS IBUKOTA JAKARTA', 'JAWA BARAT', 'JAWA TENGAH', 'DAERAH ISTIMEWA YOGYAKARTA', 'JAWA TIMUR', 'BANTEN', 'BALI', 'NUSA TENGGARA BARAT', 'NUSA TENGGARA TIMUR', 'KALIMANTAN BARAT', 'KALIMANTAN TENGAH', 'KALIMANTAN SELATAN', 'KALIMANTAN TIMUR', 'KALIMANTAN UTARA', 'SULAWESI UTARA', 'SULAWESI TENGAH', 'SULAWESI SELATAN', 'SULAWESI TENGGARA', 'GORONTALO', 'SULAWESI BARAT', 'MALUKU', 'MALUKU UTARA', 'PAPUA', 'PAPUA BARAT', 'PAPUA SELATAN', 'PAPUA TENGAH', 'PAPUA PEGUNUNGAN', 'PAPUA BARAT DAYA']


100%|██████████| 50/50 [01:18<00:00,  1.56s/it]


,alamat_clean,gedung,jalan,no,lantai,blok,kav,rt,rw,kodepos,provinsi,kode_prov,kabkota,kode_kabkota,kecamatan,kode_kec,desa,kode_desa,remaining_text
0,CYBER 2 TOWER LANTAI 26\nJL. H.R. RASUNA SAID ...,CYBER 2 TOWER JL. H.R. RASUNA SAID JAKARTA 129...,H.R. RASUNA SAID,13,26,X-5,NaN,NaN,NaN,12950,BALI,51,KABUPATEN KARANGASEM,51.07,SELAT,51.07.07,NaN,NaN,"CYBER 2 TOWER LANTAI JL. BLOK , NO. JAKARTA - ..."
6,JL. GURAMI NO. 173,NaN,GURAMI,173,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,JL. NO.
7,"MENARA KARYA LANTAI 22, JL. H.R. RASUNA SAID B...",MENARA KARYA,H.R. RASUNA SAID,NaN,22,X-5,1-2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LANTAI , JL. BLOK KAV."
8,"GEDUNG CYBER 2 TOWER LT.23, JL. HR RASUNA SAID...",GEDUNG CYBER 2 TOWER,HR RASUNA SAID,13,23,X-5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LT., JL. BLOK , NO."
9,"GD. MENARA KARYA LT.23, JL.HR. RASUNA SAID BLO...",GD. MENARA KARYA,HR. RASUNA SAID,NaN,23,X-5,1-2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LT., JL. BLOK KAV."
10,MENARA KARYA LT.23 JL.HR.RASUNA SAID BLOK X-5 ...,MENARA KARYA JL.HR.RASUNA SAID,NaN,NaN,23,X-5,1-2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MENARA KARYA LT. JL.HR.RASUNA SAID BLOK KAV.
11,"GEDUNG MENARA KARYA LT. 18, JL. H.R. RASUNA SA...",GEDUNG MENARA KARYA,H.R. RASUNA SAID,NaN,18,X-5,1-2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LT. , JL. , BLOK , KAV."
12,JL. KOM. LAUT. YOS SUDARSO (DALAM GANG PRIBADI...,NaN,KOM. LAUT. YOS SUDARSO (DALAM GANG PRIBADI LIN...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,JL.
13,RUKO GREEN GARDEN BLOK A2-10 JL. WAHIDIN SUDIR...,RUKO GREEN GARDEN JL. WAHIDIN SUDIROHUSODO,NaN,NaN,NaN,A2-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,RUKO GREEN GARDEN BLOK JL. WAHIDIN SUDIROHUSODO
14,"MENARA KARYA LT.22, JL. HR. RASUNA SAID BLK X-...",MENARA KARYA,HR. RASUNA SAID BLK X-5,NaN,22,NaN,1-2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LT., JL. , KAV."
